In [11]:
import pandas as pd
import re
from sqlalchemy import create_engine
import urllib

In [12]:
# Configurar cadena de conexión
params = urllib.parse.quote_plus(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=181.57.189.150,1433;"
    "DATABASE=Ventas_Comerssia;"
    "UID=sa;"
    "PWD=P@ssw0rd*;"
)

# Crear el motor
engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params}")

In [13]:
query_Data = """
with Clientes_DATA_Limpios AS (      
    SELECT *,
		REPLACE(LTRIM(RTRIM(Id_Cliente)),' ', '') AS Documento
    FROM Ventas_Shopify.dbo.Ventas_ShopifyMovinova  
)
SELECT DISTINCT c.Cliente,
	C.Documento,
    c.Nombres,
    c.Apellidos,
    c.Email,
    c.Celular,
    c.TipoIdentificacion
FROM Ventas_Comerssia.DBO.View_Clientes c
 INNER JOIN Clientes_DATA_Limpios v ON c.Documento = v.Documento
 --WHERE Fecha >= '2025-07-25'
"""

# Ejecutar y cargar en DataFrame
df_Data = pd.read_sql(query_Data, engine)

query_com = """
SELECT DISTINCT c.Cliente,
	c.Documento,
      c.Nombres,
      c.Apellidos,
      c.Email,
      c.Celular,
      c.TipoIdentificacion
FROM Ventas_Comerssia.DBO.View_Clientes c
 INNER JOIN Ventas_Comerssia.dbo.Ventas_Comerssia v ON c.Cliente = v.ClienteLimpio
--WHERE v.Fecha >= '2025-07-25'

"""

# Ejecutar y cargar en DataFrame
df_Comerssia = pd.read_sql(query_com, engine)

df_unificado = pd.concat([df_Data, df_Comerssia], ignore_index=True)
df_unificado = df_unificado.drop_duplicates(subset='Cliente', keep='first')

# Obtener los CLICodigos ya existentes en la tabla SQL
# clientes_existentes = pd.read_sql("SELECT Cliente FROM View_Contacto_Clientes", con=engine)

In [14]:
df_unificado

,Cliente,Documento,Nombres,Apellidos,Email,Celular,TipoIdentificacion
0,C0000000,0000000,MA. LOURDES,RESTREPO,natibecerra@gmail.com,None,CC
1,C0000000000,0000000000,DANIELA,LOPEZ,daniela.lopez.80214@gmail.com,3196463029,CC
2,C0,0,ALCIDES,ARIAS,ALCIDES.ARIASGARCIA@GMAIL.COM,+573102077625,CC
3,C095937276,095937276,GABRIELA,RODRIGUEZ,gabrielarodriguez374@gmail.com,3223881350,PA
4,C1000005625,1000005625,YESIKA,VARGAS,yesivargas95@hotmail.com,3043882312,CC
...,...,...,...,...,...,...,...
301934,CYE065325,YE065325,LUIZ,GIANESELLA,LOTAVIOGH@GMAIL.COM,UNDEFINED,PA
301935,CYI024605,YI024605,GERUSA,TEIXEIRA,NEGADO@PROVENZAL.NET,1111111111,CC
301936,CZ5012845,Z5012845,LATA,RAO,NEGADO@PROVENZAL.NET,1111111111,CC
301937,CZ6370250,Z6370250,SAMMER,KAPUR,SSKK46@GMAIL.COM,1111111111,CE


In [15]:
#Limpiar Celular
def limpiar_celular(cel):
    if pd.isna(cel):
        return ""
    cel = re.sub(r'\D', '', str(cel))  # Elimina todo lo que no sea dígito
    if cel.startswith("57") and len(cel) > 10:
        cel = cel[2:]
    return cel

df_unificado['Celular'] = df_unificado['Celular'].apply(limpiar_celular)

#Validar celular
def es_celular_valido(cel):
    return bool(re.fullmatch(r"3\d{9}", str(cel)))

df_unificado['CelularValido'] = df_unificado['Celular'].apply(es_celular_valido)

#Limpiar y validar email
df_unificado['Email'] = df_unificado['Email'].apply(lambda x: str(x).strip().lower())

def es_email_valido(email):
    email = str(email).strip().lower()

    if pd.isna(email):
        return False

    email = str(email).strip().lower()

    # Si está vacío
    if not email:
        return False
    
    # Palabras prohibidas
    palabras_no_permitidas = ["negad", "pendi"]
    if any(palabra in email for palabra in palabras_no_permitidas):
        return False
    
    # Validación de formato
    return bool(re.fullmatch(r'^[\w\.-]+@[\w\.-]+\.\w+$', email))

df_unificado['EmailValido'] = df_unificado['Email'].apply(es_email_valido)

In [16]:
# Detectar duplicados
cel_duplicados = df_unificado['Celular'].duplicated(keep=False)
email_duplicados = df_unificado['Email'].duplicated(keep=False)

# Marcar celulares duplicados como no válidos
df_unificado.loc[cel_duplicados, 'CelularValido'] = False

# Marcar correos duplicados como no válidos
df_unificado.loc[email_duplicados, 'EmailValido'] = False

In [17]:
#Agregar columna contacto
def determinar_contacto(row):
    if row['CelularValido'] and row['EmailValido']:
        return "Cel + Email"
    elif row['CelularValido']:
        return "Cel"
    elif row['EmailValido']:
        return "Email"
    else:
        return "Falso"

df_unificado['Contacto'] = df_unificado.apply(determinar_contacto, axis=1)

In [18]:
# # Filtrar para dejar solo nuevos clientes
# df_nuevos = df_unificado[~df_unificado["Cliente"].isin(clientes_existentes["Cliente"])]

In [19]:
# # Insertar solo los nuevos en SQL
# if not df_nuevos.empty:
#     df_nuevos.to_sql("View_Contacto_Clientes", con=engine, if_exists="append", index=False)
#     print(f"{len(df_nuevos)} nuevos clientes insertados.")
# else:
#     print("No hay nuevos clientes por insertar.")

In [20]:
# Exportar todos los registros con validaciones incluidas
# df_unificado.to_excel("clientes_con_validacion.xlsx", index=False)

In [21]:
df_unificado.to_sql("View_Contacto_Clientes", engine, if_exists="replace", index=False)

63